# Recursos da estimação quântica de amplitude para o VaR

E3 mantém a contagem analítica de portas T comparável à entrega anterior e a complementa com profundidade T de escalonamentos explícitos e estimação física local no Microsoft Quantum Resource Estimator (`qdk[qre]`). Não há simulação de amplitudes nem uso de hardware na nuvem.

O operador A carrega a distribuição, calcula a perda e marca os cenários abaixo do limiar. A estimação de amplitude canônica e a busca binária seguem Woerner e Egger. O erro-alvo relativo é de 1% ou 0,1% do VaR; o tempo clássico de 0,1% continua extrapolado e é identificado como tal.

| Parte | Modelo de contagem |
|---|---|
| Carga genérica | 2^(N n) rotações; excluída da estimação física por inviabilidade e crescimento exponencial |
| Cholesky | N preparações normais de 2^n rotações, mais N(N+1)/2 multiplicações por constante |
| Fator único | N+1 preparações normais, mais 2N multiplicações por constante |
| Perda | menor contagem entre aproximação linear por partes e QROM; soma no acumulador |
| Busca de quantil | bisseções sucessivas, cada uma com ceil(pi / eps_a) chamadas de Grover |

A preparação e sua inversa, a descomputação da perda e o comparador são incluídos como na contagem original. A contagem por bloco continua uma aproximação analítica; o oráculo completo não foi sintetizado.

**Profundidade e recursos físicos.** `recursos_tolerantes_falhas.py` conta circuitos Q# concretos de Toffoli (7 T/T-dagger) e de somadores Cuccaro (2b Toffolis) e propaga suas dependências para obter T-depth. Compara blocos seriais com preparações normais independentes em paralelo; correlação e perdas que compartilham controles ou acumuladores ficam em série. O QRE recebe o caminho crítico e a demanda de estados T separadamente e calcula fábricas paralelas, qubits físicos e duração com código de superfície. Esses resultados são condicionais ao modelo lógico, não uma compilação validada do VaR inteiro.

Fontes primárias: [Beverland et al.](https://arxiv.org/abs/2211.07629), [Microsoft QRE](https://github.com/microsoft/qdk/wiki/QREv3), [Cuccaro et al.](https://arxiv.org/abs/quant-ph/0410184), [QROM](https://doi.org/10.1103/PhysRevX.8.041015).


In [1]:
import math
import numpy as np
import pandas as pd

from carteira import ALPHA, K, montar_carteira, perda_por_acao, perdas
from scipy.special import ndtr

NS = [2**k for k in range(10)]
EPS = [0.01, 0.001]            # erro-alvo relativo do VaR: o do notebook clássico e um 10 vezes menor
FAIXA_Z = 4.0                  # a grade cobre +-4 desvios padrão
GUARDA = 3                     # bits extras na aritmética de ponto fixo
T_ROTACAO = 130                # portas T por rotação com erro 1e-13 (~3 log2(1/1e-13)); ~1e9 rotações por estimação
T_TOFFOLI = 7
FATOR_QAE = 1.0                # chamadas = FATOR_QAE * pi / eps_a; métodos iterativos têm constante maior
TAXAS_T = [1e4, 1e6, 1e8]      # cenários abstratos de throughput; não são estimativas físicas de hardware
M_REF = 2**17                  # cenários clássicos para medir VaR, densidade e faixa da perda

## Discretização

O circuito só vê a grade de 2^n valores por ação. Para N = 1, o VaR da distribuição discretizada é comparado com o contínuo, e n é o menor valor com erro abaixo de eps / 2.

In [2]:
def grade(n, faixa=FAIXA_Z):
    z = np.linspace(-faixa, faixa, 2**n)
    bordas = np.concatenate([[-np.inf], (z[1:] + z[:-1]) / 2, [np.inf]])
    return z, np.diff(ndtr(bordas))          # pontos e probabilidades de cada ponto


def var_discreto(z, p):
    L = perda_por_acao(z)
    ordem = np.argsort(L)
    return L[ordem][np.searchsorted(np.cumsum(p[ordem]), ALPHA)]


var_cont = var_discreto(*grade(21, faixa=8.0))     # referência: grade muito fina, praticamente contínua
erro_disc = {}
for n in range(4, 13):
    erro_disc[n] = var_discreto(*grade(n)) / var_cont - 1
    print(f"n={n}: VaR discreto / contínuo - 1 = {erro_disc[n]:+.3%}")
BITS = {eps: min(n for n, e in erro_disc.items() if abs(e) <= eps / 2) for eps in EPS}
print("qubits por ação:", BITS)

n=4: VaR discreto / contínuo - 1 = -14.755%
n=5: VaR discreto / contínuo - 1 = -10.251%
n=6: VaR discreto / contínuo - 1 = +1.476%
n=7: VaR discreto / contínuo - 1 = -1.293%
n=8: VaR discreto / contínuo - 1 = +0.135%
n=9: VaR discreto / contínuo - 1 = +0.382%
n=10: VaR discreto / contínuo - 1 = -0.194%
n=11: VaR discreto / contínuo - 1 = +0.044%
n=12: VaR discreto / contínuo - 1 = +0.073%
qubits por ação: {0.01: 8, 0.001: 11}


## Parâmetros do circuito para cada N

In [3]:
def segmentos(f_grade, tol):
    """Menor número de segmentos lineares uniformes com erro máximo <= tol na grade."""
    z = np.arange(len(f_grade))
    idx = z
    for s in range(1, len(z)):
        nos = np.unique(np.linspace(0, len(z) - 1, s + 1).round().astype(int))
        if np.abs(np.interp(idx, nos, f_grade[nos]) - f_grade).max() <= tol:
            return s
    return len(z) - 1


linhas = []
for N in NS:
    L = perdas(N, M_REF, "qmc", 0, montar_carteira(N))
    var = np.quantile(L, ALPHA)
    for eps in EPS:
        f_grade = perda_por_acao(grade(BITS[eps])[0])     # perda de uma ação em cada ponto da grade
        delta = eps * var
        h = 0.02 * var
        dens = np.mean(np.abs(L - var) <= h) / (2 * h)    # densidade da perda no VaR
        faixa = N * (f_grade.max() - f_grade.min())
        linhas.append({
            "N": N, "eps": eps, "var": var, "n": BITS[eps],
            "segmentos": segmentos(f_grade, delta / (4 * N)),       # N erros de aproximação somam <= delta/4
            "bits_acum": math.ceil(math.log2(faixa / (delta / (4 * N)))),
            "bissecoes": math.ceil(math.log2(faixa / delta)),
            "eps_a": dens * delta,
        })
par = pd.DataFrame(linhas)
par["chamadas_grover"] = np.ceil(FATOR_QAE * np.pi / par["eps_a"]).astype(int)
print(par.round(5).to_string(index=False))

  N   eps       var  n  segmentos  bits_acum  bissecoes   eps_a  chamadas_grover
  1 0.010   1.89728  8         37         11          9 0.00032             9863
  1 0.001   1.89728 11        108         14         12 0.00003            98629
  2 0.010   2.60110  8         43         12          9 0.00031            10105
  2 0.001   2.60110 11        137         16         13 0.00003           101050
  4 0.010   3.85194  8         51         14         10 0.00033             9633
  4 0.001   3.85194 11        158         17         13 0.00003            96322
  8 0.010   6.12044  8         64         15         10 0.00031            10044
  8 0.001   6.12044 11        171         18         13 0.00003           100433
 16 0.010  10.78553  8         64         16         10 0.00033             9521
 16 0.001  10.78553 11        187         19         13 0.00003            95209
 32 0.010  20.16992  8         64         17         10 0.00034             9359
 32 0.001  20.16992 11      

## Portas T e qubits

In [4]:
def somador(b):
    return 2 * b                                   # Toffolis


def mult(a, b):
    return a * somador(b)                          # registrador de a bits vezes constante, em b bits


def custo_grover(N, n, s, m):
    """Portas T de um operador de Grover (A e A dagger) para cada forma de carga, e qubits lógicos."""
    nx = n + GUARDA
    linear = s * (somador(nx) + mult(nx, m) + somador(m))         # por ação: comparações e afins
    tabela = 2**n + somador(m)                                    # por ação: consulta e soma no acumulador
    perda = N * min(linear, tabela)
    resto = T_TOFFOLI * (2 * perda + somador(m))                  # perda calculada e descalculada, comparador
    with np.errstate(over="ignore"):
        carga = {
            "generica": T_ROTACAO * np.float64(2.0) ** (N * n),
            "cholesky": T_ROTACAO * N * 2**n + T_TOFFOLI * N * (N + 1) / 2 * mult(n, nx),
            "fator unico": T_ROTACAO * (N + 1) * 2**n + T_TOFFOLI * 2 * N * mult(n, nx),
        }
    qubits = {"generica": N * n + 2 * m + 2, "cholesky": N * (n + nx) + 2 * m + 2,
              "fator unico": (N + 1) * n + N * nx + 2 * m + 2}
    return {c: (2 * (carga[c] + resto), qubits[c], carga[c] / (carga[c] + resto)) for c in carga}


tau_c = pd.read_csv("../dados/tau_c.csv")
if "eps" not in tau_c:
    tau_c["eps"] = 0.01


def referencia_classica(N, eps):
    candidatos = tau_c[tau_c["N"] == N].copy()
    medidos = candidatos[np.isclose(candidatos["eps"], eps)]
    if not medidos.empty:
        escolhido = medidos.loc[medidos["tau_c_s"].idxmin()]
        return escolhido["tau_c_s"], "medido", escolhido["metodo"]
    # Extrapola cada referência separadamente, depois escolhe a mais rápida.
    base = candidatos[np.isclose(candidatos["eps"], 0.01)].copy()
    base["tempo_no_alvo"] = base["tau_c_s"] * (eps / 0.01) ** (1 / base["inclinacao_erro"])
    escolhido = base.loc[base["tempo_no_alvo"].idxmin()]
    return escolhido["tempo_no_alvo"], "extrapolado", escolhido["metodo"]

linhas = []
for _, r in par.iterrows():
    tc, tipo_tempo, referencia = referencia_classica(r["N"], r["eps"])
    for carga, (t_grover, qubits, fracao) in custo_grover(r["N"], r["n"], r["segmentos"], r["bits_acum"]).items():
        t_total = r["bissecoes"] * r["chamadas_grover"] * t_grover
        linhas.append({"N": r["N"], "eps": r["eps"], "carga": carga, "qubits": qubits, "fracao_carga": fracao, "T_por_grover": t_grover,
                       "T_total": t_total, "tau_c_s": tc, "taxa_T_empate": t_total / tc,
                       "n": int(r["n"]), "segmentos": int(r["segmentos"]), "bits_acum": int(r["bits_acum"]),
                       "bissecoes": int(r["bissecoes"]), "chamadas_grover": int(r["chamadas_grover"]),
                       "tipo_tempo_classico": tipo_tempo, "referencia_classica": referencia,
                       **{f"tau_q_s_{t:.0e}": t_total / t for t in TAXAS_T}})
res = pd.DataFrame(linhas)
res.to_csv("../dados/tau_q.csv", index=False)
print(res[res["carga"] != "generica"][["N", "eps", "carga", "qubits", "fracao_carga", "T_total", "tau_c_s", "taxa_T_empate"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3g}"))

  N   eps       carga   qubits  fracao_carga  T_total  tau_c_s  taxa_T_empate
  1  0.01    cholesky       43         0.895 6.85e+09  0.00043       1.59e+13
  1  0.01 fator unico       51         0.945  1.3e+10  0.00043       3.02e+13
  1 0.001    cholesky       55         0.902 7.05e+11   0.0045       1.57e+14
  1 0.001 fator unico       66         0.948 1.34e+12   0.0045       2.98e+14
  2  0.01    cholesky       64         0.898 1.42e+10  0.00406       3.51e+12
  2  0.01 fator unico       72         0.929 2.05e+10  0.00406       5.05e+12
  2 0.001    cholesky       84         0.902 1.57e+12   0.0678       2.31e+13
  2 0.001 fator unico       95         0.932 2.27e+12   0.0678       3.35e+13
  4  0.01    cholesky      106           0.9 3.11e+10   0.0157       1.98e+12
  4  0.01 fator unico      114         0.916 3.71e+10   0.0157       2.35e+12
  4 0.001    cholesky      136         0.903 3.01e+12     1.31       2.31e+12
  4 0.001 fator unico      147          0.92 3.67e+12     1.31  

## Requisito de throughput

`taxa_T_empate = T_total / tau_c_s` é o throughput médio necessário para consumir todas as portas T dentro do prazo clássico. Não determina sozinho o tempo físico: o caminho crítico e as fábricas de estados mágicos são tratados na próxima etapa. Tempos clássicos em 0,1% são extrapolados, não medidos.

As duas figuras finais do artigo são geradas após E4 com `python codigo/figuras_artigo.py`; as curvas exponenciais de carga genérica não entram no eixo principal.


In [5]:
for t in TAXAS_T:
    v = res[res[f"tau_q_s_{t:.0e}"] < res["tau_c_s"]]
    print(f"{t:.0e} T/s: vantagem de throughput em {len(v)} de {len(res)} combinações")


1e+04 T/s: vantagem de throughput em 0 de 60 combinações
1e+06 T/s: vantagem de throughput em 0 de 60 combinações
1e+08 T/s: vantagem de throughput em 0 de 60 combinações


## Profundidade T e estimação física tolerante a falhas

Os próximos resultados usam QDK local, com tempos físicos de porta e medição de 50 ns e 100 ns, erro físico de 10^-4 ou 10^-3, código de superfície e fábricas RoundBasedFactory em paralelo. O orçamento de falha física é 1%; ele não substitui os orçamentos de erro estatístico, discretização e síntese do algoritmo.

A largura adiciona ao modelo anterior o registrador de fase da estimação canônica e workspace de iteração unária da QROM; o layout PSSPC e as fábricas são contabilizados depois. A profundidade é a dos escalonamentos declarados, não uma alegação de profundidade ótima do VaR completo. A síntese e a contagem de primitivas são auditadas; a composição do oráculo ainda depende do modelo analítico.

Saídas: `recursos_primitivas.csv`, `recursos_logicos.csv`, `recursos_fisicos.csv` (todos os pontos Pareto encontrados) e `recursos_premissas.json`. Fila, rede e preço comercial não entram no modelo físico.


In [6]:
from recursos_tolerantes_falhas import executar

logicos, fisicos = executar()
print(logicos[["N", "eps", "carga", "escalonamento", "T_total", "T_depth_escalonamento"]].to_string(index=False))
mais_rapidos = fisicos.loc[fisicos.groupby(["N", "eps", "carga", "escalonamento", "erro_porta_fisica"])["runtime_s"].idxmin()]
print(mais_rapidos[["N", "eps", "carga", "escalonamento", "erro_porta_fisica", "runtime_s", "qubits_fisicos", "fabricas_T_paralelas"]].to_string(index=False))


QRE: concluído erro físico 0.0001
QRE: concluído erro físico 0.001
  N   eps       carga     escalonamento          T_total  T_depth_escalonamento
  1 0.010    cholesky            serial       6845355972             6443774064
  1 0.010    cholesky normais_paralelas       6845355972             6443774064
  1 0.010 fator unico            serial      12972409380            12477089520
  1 0.010 fator unico normais_paralelas      12972409380             6568758000
  1 0.001    cholesky            serial     704580326976           672709746432
  1 0.001    cholesky normais_paralelas     704580326976           672709746432
  1 0.001 fator unico            serial    1339899424992          1305841647744
  1 0.001 fator unico normais_paralelas    1339899424992           675626008704
  2 0.010    cholesky            serial      14235438960            13323078720
  2 0.010    cholesky normais_paralelas      14235438960             7269779520
  2 0.010 fator unico            serial      20512826

## Hipóteses reformuladas

As hipóteses H1 e H2 são avaliadas contra Monte Carlo e quasi-Monte Carlo separadamente; o arquivo `dados/hipoteses.csv` registra coeficientes, mínimos e contraexemplos. A precisão de 0,1% mantém a referência clássica extrapolada.


In [7]:
from avaliar_hipoteses import avaliar

print(avaliar().to_string(index=False))


hipotese referencia  inclinacao_cholesky  inclinacao_fator_unico  minimo_r_estrela  contraexemplos  satisfeita_no_modelo
      H1         mc             0.410635               -0.117335               NaN             NaN                  True
      H2         mc                  NaN                     NaN      2.125318e+11             0.0                  True
      H1        qmc             0.453495               -0.074475               NaN             NaN                  True
      H2        qmc                  NaN                     NaN      7.116307e+11             0.0                  True
